# NumPy Pipeline Executor — Validation

Validates that `NumpyPipelineExecutor` produces **bit-identical results** to Spark
for every row in `notebooks/out/2026_01_01_001.parquet`.

## What this notebook does

1. Load `notebooks/out/2026_01_01_001.parquet` with pandas.
2. Start a local Spark session and fit `dsl_001.yaml` on the data.
3. Save the fitted pipeline to a temp dir (`stages.json` + `config.json`).
4. Load it with `NumpyPipelineExecutor`.
5. Compare Spark output vs NumPy output for every row (14 feature columns).
6. Print a summary.

In [1]:
import os, sys, tempfile, math
import numpy as np
import pandas as pd

# Make sure src/ is on the path
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print('repo_root:', repo_root)

repo_root: /home/jorge/DocumentsWLS/Data_Science_Projects/Kubernetes-MLOPS-platform


## 1  Load parquet with pandas

In [2]:
PARQUET_PATH = os.path.join(repo_root, 'notebooks', 'out', '2026_01_01_001.parquet')
DSL_YAML     = os.path.join(repo_root, 'k3s', 'spark', 'preprocess', 'dsl_001.yaml')

pdf = pd.read_parquet(PARQUET_PATH)
print(f'Rows: {len(pdf):,}  Columns: {list(pdf.columns)}')
pdf.head(3)

Rows: 3,000  Columns: ['src_port', 'dst_port', 'protocol', 'packet_count', 'conn_state', 'bytes_transferred', 'timestamp', 'attack']


,src_port,dst_port,protocol,packet_count,conn_state,bytes_transferred,timestamp,attack
0,36060,16998,TCP,301,EST,317008.473306,2026-01-01 00:30:03,0
1,51170,14275,UDP,382,EST,92432.128652,2026-01-01 00:30:24,0
2,64602,869,TCP,280,EST,5424.272789,2026-01-01 00:30:42,0


## 2  Start local Spark and fit the DSL pipeline

In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master('local[2]')
    .appName('dsl_numpy_validation')
    .config('spark.sql.session.timeZone', 'UTC')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)

JAVA_HOME is not set


PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [ ]:
from src.dsl.pipeline import Pipeline

sdf = spark.createDataFrame(pdf)
print('Spark schema:')
sdf.printSchema()

pipeline = Pipeline.from_yaml(DSL_YAML)
fitted   = pipeline.fit(sdf)
print('Pipeline fitted successfully.')

## 3  Save pipeline artifacts and get Spark output

In [ ]:
artifact_dir = tempfile.mkdtemp(prefix='dsl_artifacts_')
fitted.save(artifact_dir)
print('Artifacts saved to:', artifact_dir)
print('Files:', os.listdir(artifact_dir))

In [ ]:
import yaml as _yaml

with open(DSL_YAML) as f:
    dsl_cfg = _yaml.safe_load(f)

feature_cols = dsl_cfg['final_features']['features']
print(f'{len(feature_cols)} feature columns:', feature_cols)

# Spark: transform + select only feature columns
spark_out = fitted.transform(sdf).select(*feature_cols)
spark_pdf = spark_out.toPandas()
print('Spark output shape:', spark_pdf.shape)
spark_pdf.head(3)

## 4  Load NumpyPipelineExecutor and transform each row

In [ ]:
from src.dsl.numpy_executor import NumpyPipelineExecutor

executor = NumpyPipelineExecutor.from_dir(artifact_dir)
print('Executor loaded. Feature order:', executor.final_features)

In [ ]:
numpy_rows = []

for _, row in pdf.iterrows():
    row_dict = row.to_dict()
    vec = executor.transform_to_vector(row_dict)
    numpy_rows.append(vec)

numpy_pdf = pd.DataFrame(numpy_rows, columns=feature_cols)
print('NumPy output shape:', numpy_pdf.shape)
numpy_pdf.head(3)

## 5  Compare Spark vs NumPy — numerical tolerance

In [ ]:
# Reset indices so comparison is aligned
spark_pdf  = spark_pdf.reset_index(drop=True).astype(float)
numpy_pdf  = numpy_pdf.reset_index(drop=True).astype(float)

RTOL = 1e-5   # relative tolerance
ATOL = 1e-7   # absolute tolerance

mismatches = {}
for col in feature_cols:
    sp = spark_pdf[col].values
    np_ = numpy_pdf[col].values
    close = np.allclose(sp, np_, rtol=RTOL, atol=ATOL, equal_nan=True)
    if not close:
        diff = np.abs(sp - np_)
        mismatches[col] = {'max_abs_diff': diff.max(), 'mean_abs_diff': diff.mean()}

if not mismatches:
    print(f'✓ ALL {len(feature_cols)} feature columns match within rtol={RTOL}, atol={ATOL}')
    print(f'  Checked {len(pdf):,} rows.')
else:
    print(f'✗ Mismatches found in {len(mismatches)} column(s):')
    for col, stats in mismatches.items():
        print(f'  {col}: max_abs_diff={stats["max_abs_diff"]:.2e}, mean={stats["mean_abs_diff"]:.2e}')

## 6  Per-column max absolute difference

In [ ]:
summary = pd.DataFrame({
    'max_abs_diff': [
        np.abs(spark_pdf[c].values - numpy_pdf[c].values).max()
        for c in feature_cols
    ],
    'max_rel_diff': [
        np.abs(
            (spark_pdf[c].values - numpy_pdf[c].values)
            / np.where(np.abs(spark_pdf[c].values) > 1e-10, spark_pdf[c].values, 1)
        ).max()
        for c in feature_cols
    ],
}, index=feature_cols)

print(summary.to_string())

## 7  Test the online path: raw_event_to_features → executor

In [ ]:
from src.converters.raw_to_features import raw_event_to_features

# Simulate a raw Kafka event for the first row of the parquet
first = pdf.iloc[0]
raw_event = {
    'timestamp': str(first['timestamp']),
    'event_id':  'test-uuid-001',
    'properties': {
        'src_port':          int(first['src_port']),
        'dst_port':          int(first['dst_port']),
        'protocol':          str(first['protocol']),
        'packet_count':      int(first['packet_count']),
        'conn_state':        str(first['conn_state']),
        'bytes_transferred': float(first['bytes_transferred']),
    },
}

print('Raw event:', raw_event)
print()

feature_row = raw_event_to_features(raw_event)
print('After raw_event_to_features:')
for k, v in feature_row.items():
    print(f'  {k}: {v!r}')

In [ ]:
online_vec  = executor.transform_to_vector(feature_row)
spark_vec   = spark_pdf.iloc[0].tolist()

print(f'Online (NumPy) vector ({len(online_vec)} features):')
for col, v in zip(feature_cols, online_vec):
    print(f'  {col:25s} {v:+.6f}')

print()
print(f'Spark vector ({len(spark_vec)} features):')
for col, v in zip(feature_cols, spark_vec):
    print(f'  {col:25s} {v:+.6f}')

print()
diffs = [abs(a - b) for a, b in zip(online_vec, spark_vec)]
print(f'Max absolute difference (row 0): {max(diffs):.2e}')

if max(diffs) < 1e-4:
    print('✓ Online path matches Spark output for row 0.')
else:
    print('✗ Discrepancy detected — check timestamp timezone handling.')

## 8  Latency benchmark (single-row NumPy vs overhead)

In [ ]:
import timeit

BENCHMARK_ROWS = 200
sample_rows = [pdf.iloc[i % len(pdf)].to_dict() for i in range(BENCHMARK_ROWS)]

def run_batch():
    for row in sample_rows:
        executor.transform_to_vector(row)

n_reps = 20
total  = timeit.timeit(run_batch, number=n_reps)
per_row_us = total / (n_reps * BENCHMARK_ROWS) * 1e6

print(f'NumPy executor latency:')
print(f'  {per_row_us:.1f} µs / row  ({1e6/per_row_us:,.0f} rows/sec)')
print(f'  (measured over {n_reps * BENCHMARK_ROWS:,} calls)')

In [ ]:
spark.stop()
print('Done.')